In [1]:
!pip install -U minsearch qdrant_client

In [2]:
import requests
import pandas as pd

url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

In [3]:
from tqdm.auto import tqdm

def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

In [4]:
text_fields=["question", "section", "text"],
keyword_fields=["course", "id"]

In [5]:
boost = {'question': 1.5, 'section': 0.1}

In [6]:
from minsearch import VectorSearch

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

In [8]:
texts = []

for doc in documents:
    t = doc['question']
    texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

## Q1. Minsearch text

Now let's evaluate our usual minsearch approach, indexing documents with:
```python
text_fields=["question", "section", "text"],
keyword_fields=["course", "id"]
```
but tweak the parameters for search. Let's use the following boosting params:

```python
boost = {'question': 1.5, 'section': 0.1}
```

What's the hitrate for this approach?

* 0.64
* 0.74
* 0.84
* 0.94

In [2]:
import requests  # for downloading datasets
import pandas as pd  # for loading and handling tabular data
from minsearch import Index  # the MinSearch class for text-based search
from tqdm.auto import tqdm  # for progress bars during evaluation

In [3]:
# Define base URL to GitHub raw data
url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'

# URL for the documents JSON file
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'

# Download and parse the documents
documents = requests.get(docs_url).json()

# URL for the ground truth CSV file
ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'

# Load ground truth into a DataFrame
df_ground_truth = pd.read_csv(ground_truth_url)

# Convert DataFrame to list of dictionaries for easier processing
ground_truth = df_ground_truth.to_dict(orient='records')

In [4]:
# Compute Hit Rate: % of queries for which the correct document was retrieved
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:  # If any retrieved doc matches the correct ID
            cnt = cnt + 1
    return cnt / len(relevance_total)

# Compute Mean Reciprocal Rank (MRR)
def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:  # True means relevant doc found at this rank
                total_score = total_score + 1 / (rank + 1)
                break  # only the first correct hit counts for MRR
    return total_score / len(relevance_total)

# Main evaluation loop
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):  # iterate over each query
        doc_id = q['document']  # correct document id
        results = search_function(q)  # run search
        relevance = [d['id'] == doc_id for d in results]  # check if results match the true id
        relevance_total.append(relevance)  # collect all relevance flags

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

In [5]:
# Create a MinSearch index with specified text and keyword fields
index = Index(
    text_fields=["question", "section", "text"],  # full-text searchable fields
    keyword_fields=["course", "id"]  # fields for exact matching
)

# Fit the index to our document list
index.fit(documents)

In [6]:
def search_function(q):
    return index.search(
        q["question"],  # use the question as the search query
        filter_dict={"course": q["course"]},  # filter by course to narrow down results
        boost_dict={"question": 1.5, "section": 0.1},  # boost weights for fields
        num_results=5  # how many top results to return
    )


In [7]:
# Evaluate using the ground truth and the defined search function
results = evaluate(ground_truth, search_function)

# Print final evaluation metrics
print(results)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.848714069591528, 'mrr': 0.7283553058137033}


## Q2. Vector search for question

Now let's index these embeddings with minsearch:

```python
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)
```

Evaluate this seach method. What's MRR for it?

- 0.25
- 0.35
- 0.45
- 0.55

In [15]:
from minsearch import VectorSearch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

texts = [doc['question'] for doc in documents]

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

X = pipeline.fit_transform(texts)

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

def vector_search_function(q):
    query_vec = pipeline.transform([q["question"]])
    return vindex.search(
        query_vector=query_vec[0],
        filter_dict={"course": q["course"]},
        num_results=5
    )

results = evaluate(ground_truth, vector_search_function)
print(results)


  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.48195374972984656, 'mrr': 0.3568763057416613}


## Q3. Vector search for question and answer

We only used question in Q2. We can use both question and answer:

```python
texts = []

for doc in documents:
    t = doc['question'] + ' ' + doc['text']
    texts.append(t)
```

Using the same pipeline (`min_df=3 for TF-IDF vectorizer and `n_components=128` for SVD), evaluate the performance of this
approach

What's the hitrate?

- 0.62
- 0.72
- 0.82
- 0.92


In [16]:
# --- Build Combined Texts (Question + Answer) ---
texts = [doc['question'] + ' ' + doc['text'] for doc in documents]

# --- Build Embedding Pipeline ---
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

X = pipeline.fit_transform(texts)

# --- Vector Search Index ---
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

def vector_search_combined(q):
    query_vec = pipeline.transform([q["question"]])
    return vindex.search(
        query_vector=query_vec[0],
        filter_dict={"course": q["course"]},
        num_results=5
    )

# --- Evaluate ---
results = evaluate(ground_truth, vector_search_combined)
print(results)

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.8210503566025502, 'mrr': 0.6711944384410349}


## Q4. Qdrant

Now let's evaluate the following settings in Qdrant:

- `text = doc['question'] + ' ' + doc['text']`
- `model_handle = "jinaai/jina-embeddings-v2-small-en"`
- `limit = 5`

What's the MRR?

- 0.65
- 0.75
- 0.85
- 0.95

In [8]:
!pip install sentence_transformers

In [21]:
import requests
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from sklearn.metrics.pairwise import cosine_similarity

# --- Load data ---
url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'

documents = requests.get(url_prefix + 'search_evaluation/documents-with-ids.json').json()
df_ground_truth = pd.read_csv(url_prefix + 'search_evaluation/ground-truth-data.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

# --- Embed text using Jina Embeddings ---
model = SentenceTransformer("jinaai/jina-embeddings-v2-small-en")

texts = [doc['question'] + " " + doc['text'] for doc in documents]
vectors = model.encode(texts, show_progress_bar=True)

# --- Initialize Qdrant (in-memory) ---
client = QdrantClient(":memory:")

# Create collection
client.recreate_collection(
    collection_name="docs",
    vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE)
)

# Prepare and upload points
points = [
    PointStruct(
        id=i,
        vector=vec,
        payload={**doc, "idx": i}
    )
    for i, (vec, doc) in enumerate(zip(vectors, documents))
]

client.upsert(collection_name="docs", points=points)

# --- Define evaluation helpers ---
def hit_rate(relevance_total):
    return sum(True in line for line in relevance_total) / len(relevance_total)

def mrr(relevance_total):
    total = 0.0
    for line in relevance_total:
        for rank, rel in enumerate(line):
            if rel:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

# --- Define Qdrant Search Function ---
def qdrant_search(q):
    query_vec = model.encode([q['question']])[0]
    result = client.search(
        collection_name="docs",
        query_vector=query_vec,
        limit=5,
        query_filter=Filter(
            must=[FieldCondition(key="course", match=MatchValue(value=q["course"]))]
        )
    )
    return [r.payload for r in result]

# --- Run Evaluation ---
results = evaluate(ground_truth, qdrant_search)
print(results)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


: 

In [9]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from sentence_transformers import SentenceTransformer

# Load embedding model
#model = SentenceTransformer("jinaai/jina-embeddings-v2-small-en")
model = SentenceTransformer("jinaai/jina-embeddings-v2-small-en", 
                            trust_remote_code=True,
                            device="cpu")  # Use 'cuda' if GPU is available


# Initialize Qdrant in memory (use 'http://localhost:6333' if you have a server)
client = QdrantClient(":memory:")
COLLECTION_NAME = "faq_data"

# Create embeddings from concatenated "question + text"
texts = [doc["question"] + " " + doc["text"] for doc in documents]
vectors = model.encode(texts).tolist()

# Create collection with vector settings
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE)
)

# Upload all documents as points to Qdrant
points = [
    PointStruct(
        id=i,
        vector=vectors[i],
        payload={
            "id": doc["id"],
            "course": doc["course"],
            "question": doc["question"],
            "text": doc["text"]
        }
    )
    for i, doc in enumerate(documents)
]
client.upsert(collection_name=COLLECTION_NAME, points=points)

# Define the Qdrant vector search function
def qdrant_search_function(q):
    query_vector = model.encode(q["question"]).tolist()
    course_filter = Filter(
        must=[FieldCondition(key="course", match=MatchValue(value=q["course"]))]
    )
    hits = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        query_filter=course_filter,
        limit=5
    )
    return [
        {
            "id": hit.payload["id"],
            "question": hit.payload["question"],
            "text": hit.payload["text"],
            "course": hit.payload["course"]
        }
        for hit in hits
    ]

# Evaluate using provided evaluate() function
results = evaluate(ground_truth, qdrant_search_function)
print(results)


: 

## Q5. Cosine simiarity

In the second part of the module, we looked at evaluating
the entire RAG approach. In particular, we looked at 
comparing the answer generated by our system with the actual
answer from the FAQ.

One of the ways of doing it is using the cosine similarity. 
Let's see how to calculate it.

Cosine similarity is a dot product between two normalized vectors.
In geometrical sense, it's the cosine of the angle between
the vectors. Look up "cosine similarity geometry" if you want to
learn more about it.

For us, it means that we need two things:

- First, we normalize each of the vectors
- Then, compute the dot product

So, we get this:

```python
def cosine(u, v):
    u = normalize(u)
    v = normalize(v)
    return u.dot(v)
```

For normalization, we first compute the vector norm (its length),
and then divide the vector by it:

```python
def normalize(u):
    norm = np.sqrt(u.dot(u))
    return u / norm
```

(where `np` is `import numpy as np`)

Or we can simplify it:

```python
def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)
```

Now let's use this function to compute the
A->Q->A cosine similarity.

We will use the results from [our gpt-4o-mini evaluations](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/03-evaluation/rag_evaluation/data/results-gpt4o-mini.csv):


```python
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)
```


When creating embeddings, we will use a simple way -
the same we used in the [Embeddings](#embeddings) section:

```python
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
```

Let's fit the vectorizer on all the text data we have:

```python
pipeline.fit(df_results.answer_llm + ' ' + df_results.answer_orig + ' ' + df_results.question)
```

Now use the `transform` methon of the pipeline to create the embeddings and calculate the cosine similarity between each
pair.

What's the average cosine?

- 0.64
- 0.74
- 0.84
- 0.94

This is how you do it:

- For each answer pair, compute
    - `v_llm` for the answer from the LLM 
    - `v_orig` for the original answer
    - then compute the cosine between them
- At the end, take the average



In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

# Cosine similarity function
def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)

# Load results
url_prefix = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/"
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

# Build TF-IDF + SVD pipeline
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Fit on combined text
pipeline.fit(df_results['answer_llm'] + ' ' + df_results['answer_orig'] + ' ' + df_results['question'])

# Compute cosine similarity for each answer pair
cosines = []

for _, row in df_results.iterrows():
    v_llm = pipeline.transform([row['answer_llm']])[0]
    v_orig = pipeline.transform([row['answer_orig']])[0]
    sim = cosine(v_llm, v_orig)
    cosines.append(sim)

# Average cosine similarity
avg_cosine = np.mean(cosines)
print("Average cosine similarity:", round(avg_cosine, 4))


Average cosine similarity: 0.8416


## Q6. Rouge

And alternative way to see how two texts are similar is ROUGE. 

This is a set of metrics that compares two answers based on the overlap of n-grams, word sequences, and word pairs.

It can give a more nuanced view of text similarity than just cosine similarity alone.

We don't need to implement it ourselves, there's a python package for it:

```bash
pip install rouge
```

(The latest version at the moment of writing is `1.0.1`)

Let's compute the ROUGE score between the answers at the index 10 of our dataframe (`doc_id=5170565b`)

```
from rouge import Rouge
rouge_scorer = Rouge()

r = df_results.iloc[10]
scores = rouge_scorer.get_scores(r.answer_llm, r.answer_orig)[0]
scores
```

There are three scores: `rouge-1`, `rouge-2` and `rouge-l`, and precision, recall and F1 score for each.

* `rouge-1` - the overlap of unigrams,
* `rouge-2` - bigrams,
* `rouge-l` - the longest common subsequence

For the 10th document, Rouge-1 F1 score is 0.45

Let's compute it for the pairs in the entire dataframe.
What's the average Rouge-1 F1?

- 0.25
- 0.35
- 0.45
- 0.55


In [3]:
!pip install rouge

In [5]:
from rouge import Rouge
import pandas as pd

# Load data
url_prefix = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/"
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

# Initialize Rouge scorer
rouge_scorer = Rouge()

# Compute ROUGE-1 F1 score for each pair
rouge_1_f1_scores = []

for _, row in df_results.iterrows():
    try:
        score = rouge_scorer.get_scores(row['answer_llm'], row['answer_orig'])[0]
        rouge_1_f1 = score["rouge-1"]["f"]
        rouge_1_f1_scores.append(rouge_1_f1)
    except ValueError:
        # Handles rare empty input cases
        continue

# Average ROUGE-1 F1 score
avg_rouge_1_f1 = sum(rouge_1_f1_scores) / len(rouge_1_f1_scores)
print("Average ROUGE-1 F1 Score:", round(avg_rouge_1_f1, 4))


Average ROUGE-1 F1 Score: 0.3517


In [6]:
import pandas as pd
from rouge import Rouge

# Download the results CSV file
results_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/rag_evaluation/data/results-gpt4o-mini.csv"
df_results = pd.read_csv(results_url)

# Initialize ROUGE scorer
rouge_scorer = Rouge()

# Store all Rouge-1 F1 scores here
rouge_1_f1_scores = []

# Iterate through each row and compute rouge-1 F1 score
for _, row in df_results.iterrows():
    answer_llm = str(row['answer_llm']).strip()
    answer_orig = str(row['answer_orig']).strip()

    # Only compute if both answers are non-empty
    if answer_llm and answer_orig:
        scores = rouge_scorer.get_scores(answer_llm, answer_orig)[0]
        rouge_1_f1 = scores['rouge-1']['f']
        rouge_1_f1_scores.append(rouge_1_f1)

# Compute average Rouge-1 F1
average_rouge_1_f1 = sum(rouge_1_f1_scores) / len(rouge_1_f1_scores)

print(f"Average ROUGE-1 F1 score: {average_rouge_1_f1:.2f}")


Average ROUGE-1 F1 score: 0.35
